# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [111]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [134]:
# TODO: Import the necessary libs
# For example: 
# import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [113]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from tavily import TavilyClient
from lib.state_machine import StateMachine
from lib.agents import Agent
from lib.llm import LLM
from lib.parsers import PydanticOutputParser
import operator
from pydantic import BaseModel, Field

In [135]:
# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
 

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web



#### Retrieve Game Tool

In [140]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

 
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")

@tool
def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds most results in the vector DB.
    Args:
    - query: A question about the game industry.
    Returns a list of games with Platform, Name, YearOfRelease, Description.
    """
    results = collection.query(query_texts=[query], n_results=5)
    return [
        {
            "Platform": results["metadatas"][0][i].get("Platform"),
            "Name": results["metadatas"][0][i].get("Name"),
            "YearOfRelease": results["metadatas"][0][i].get("YearOfRelease"),
            "Description": results["metadatas"][0][i].get("Description"),
        }
        for i in range(len(results["ids"][0]))
    ]

results = retrieve_game("mario")
assert len(results) > 0, "No results returned"
assert "Name" in results[0], "Missing Name field"
print("✓ retrieve_game passed ✅")

✓ retrieve_game passed ✅


In [137]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from openai import OpenAI

#### Evaluate Retrieval Tool

In [141]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

class EvaluationReport(BaseModel):
    """Evaluation of retrieved documents"""
    useful: bool = Field(description="Whether the documents are useful to answer the question")
    description: str = Field(description="Detailed explanation about the evaluation result")

 

@tool
def evaluate_retrieval(question: str, retrieved_docs: list) -> dict:
    """
    Evaluates whether retrieved documents are sufficient to answer the question.
    Args:
    - question: Original question from the user.
    - retrieved_docs: Retrieved documents most similar to the user query in the Vector Database.
    Returns useful (bool) and description (str).
    """
    llm = LLM(
        model="gpt-4o-mini",
        api_key=os.environ.get("OPENAI_API_KEY")
    )

    prompt = f"""Your task is to evaluate if the documents are enough to respond the query.
Give a detailed explanation, so it's possible to take an action to accept it or not.
Question: {question}
Retrieved Documents: {retrieved_docs}
Respond ONLY with a JSON object with fields: "useful" (bool) and "description" (str).
"""
    response = llm.invoke(input=prompt, response_format=EvaluationReport)
    parser = PydanticOutputParser(model_class=EvaluationReport)
    evaluation = parser.parse(response)
    return {"useful": evaluation.useful, "description": evaluation.description}



# Test
result = evaluate_retrieval(
    question="What is a good mario game?",
    retrieved_docs=retrieve_game("mario")
)
assert "useful" in result, "Missing useful field"
assert "description" in result, "Missing description field"
assert isinstance(result["useful"], bool), "useful must be a bool"
print("✓ evaluate_retrieval passed ✅")


✓ evaluate_retrieval passed ✅


#### Game Web Search Tool

In [142]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

@tool
def game_web_search(question: str) -> list:
    """
    Searches the web for game industry information.
    Args:
    - question: A question about the game industry.
    Returns a list of results with title, url, and content.
    """
    search_results = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY")).search(question, max_results=4)
    return [
        {"title": r["title"], "url": r["url"], "content": r["content"]}
        for r in search_results["results"]
    ]

# Test
results = game_web_search("mario games history")
assert len(results) > 0, "No results returned"
assert "title" in results[0], "Missing title field"
print("✓ game_web_search passed ✅")

✓ game_web_search passed ✅


### Agent

In [143]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

udaplay_agent = Agent(
    model_name="gpt-4o-mini",
    instructions="""You are UdaPlay, an AI Research Agent specialized in the video game industry.

Your role is to help users find information about video games by:
1. First searching your internal knowledge base using retrieve_game
2. Evaluating if the retrieved information is sufficient using evaluate_retrieval
3. If insufficient, searching the web using game_web_search
4. Providing clear, accurate, and well-cited answers

Always follow this workflow:
- Start by retrieving from the internal database
- Evaluate if the retrieved documents are useful
- Only search the web if internal knowledge is insufficient
- Cite your sources (Internal DB or web URLs) in your final answer""",
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.7
)

# Test
assert udaplay_agent is not None, "Agent not created"
assert len(udaplay_agent.tools) == 3, "Expected 3 tools"

In [144]:
questions = [
    "When were Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for Playstation 5?",
]

session_id = "test_session"

for i, query in enumerate(questions, 1):
    print(f"\n{'━'*60}")
    print(f"  Query {i} of {len(questions)}")
    print(f"  {query}")
    print(f"{'━'*60}")

    run = udaplay_agent.invoke(query, session_id=session_id)
    final_state = run.get_final_state()
    messages = final_state.get("messages", [])

    # Tool usage trace
    sources = []
    print("\n  Reasoning:")
    for msg in messages:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                print(f"    → [{tc.function.name}] {list(args.values())[0]}")
                if tc.function.name == "retrieve_game":
                    sources.append("Internal Game Database (ChromaDB)")
                elif tc.function.name == "game_web_search":
                    for m in messages:
                        if isinstance(m, ToolMessage) and m.name == "game_web_search":
                            try:
                                results = json.loads(json.loads(m.content))
                                sources += [r.get("url") for r in results if r.get("url")]
                            except:
                                pass

    # Final answer
    final = next(
        (m.content for m in reversed(messages)
         if isinstance(m, AIMessage) and m.content and not m.tool_calls),
        "No answer generated."
    )

    print(f"\n  Answer:\n  {final}")

    # Citations
    if sources:
        print(f"\n  Citations:")
        for idx, src in enumerate(set(sources), 1):
            print(f"    [{idx}] {src}")

    print(f"\n  Tokens used: {final_state.get('total_tokens', 0)}")

print(f"\n{'━'*60}")
print(f"  REPORT SUMMARY")
print(f"{'━'*60}")
print(f"  Session : {session_id}")
print(f"  Queries : {len(questions)} completed")
print(f"  Tools   : retrieve_game → evaluate_retrieval → game_web_search")
print(f"  Workflow: internal DB first, web fallback if insufficient")
print(f"{'━'*60}\n")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Query 1 of 3
  When were Pokémon Gold and Silver released?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

  Reasoning:
    → [retrieve_game] Pokémon Gold and Silver release date
    → [evaluate_retrieval] When were Pokémon Gold and Silver released?

  Answer:
  Pokémon Gold and Silver were released in 1999 for the Game Boy Color. These games are part of the second generation of Pokémon and introduced new regions, Pokémon, and gameplay mechanics.

  Citations:
    [1] Internal Game Database (ChromaDB)

  Tokens used: 2446

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### (Optional) Advanced

In [147]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes

from lib.memory import LongTermMemory, MemoryFragment
from lib.vector_db import VectorStoreManager
from typing import TypedDict, List, Optional
from lib.state_machine import StateMachine, Step, EntryPoint, Termination, Run

# ── Long-term memory ───────────────────────────────────────
vsm = VectorStoreManager(openai_api_key=os.environ.get("OPENAI_API_KEY"))
long_term = LongTermMemory(db=vsm)

# ── State ──────────────────────────────────────────────────
class UdaPlayState(TypedDict):
    user_query: str
    instructions: str
    messages: List
    current_tool_calls: Optional[List]
    total_tokens: int
    session_id: str

INSTRUCTIONS = """You are UdaPlay, an AI game research agent.
1. Use retrieve_game to search internal DB
2. Use evaluate_retrieval to assess results
3. Use game_web_search if results are insufficient
4. Cite your sources in the final answer"""

tools = [retrieve_game, evaluate_retrieval, game_web_search]
llm  = LLM(model="gpt-4o-mini", tools=tools, api_key=os.environ.get("OPENAI_API_KEY"))

# ── Step functions ─────────────────────────────────────────
def prepare(state: UdaPlayState) -> dict:
    messages = state.get("messages", []) or [SystemMessage(content=state["instructions"])]
    memories = long_term.search(state["user_query"], owner=state["session_id"], limit=3)
    if memories.fragments:
        context = "\n".join(f"- {f.content}" for f in memories.fragments)
        messages.append(SystemMessage(content=f"Relevant past context:\n{context}"))
    messages.append(UserMessage(content=state["user_query"]))
    return {"messages": messages}

def call_llm(state: UdaPlayState) -> dict:
    response = llm.invoke(state["messages"])
    tokens = state.get("total_tokens", 0) + (response.token_usage.total_tokens if response.token_usage else 0)
    return {
        "messages": state["messages"] + [AIMessage(content=response.content, tool_calls=response.tool_calls)],
        "current_tool_calls": response.tool_calls or None,
        "total_tokens": tokens,
    }

def run_tool(state: UdaPlayState) -> dict:
    call = state["current_tool_calls"][0]
    tool = next(t for t in tools if t.name == call.function.name)
    result = tool(**json.loads(call.function.arguments))
    return {
        "messages": state["messages"] + [ToolMessage(
            content=json.dumps(str(result)),
            tool_call_id=call.id,
            name=call.function.name
        )],
        "current_tool_calls": state["current_tool_calls"][1:] or None,
    }

def save_memory(state: UdaPlayState) -> dict:
    final = next(
        (m.content for m in reversed(state["messages"])
         if isinstance(m, AIMessage) and m.content and not m.tool_calls),
        None
    )
    if final:
        long_term.register(MemoryFragment(
            content=f"Q: {state['user_query']}\nA: {final}",
            owner=state["session_id"]
        ))
    return {}

# ── Routing ────────────────────────────────────────────────
def route(state: UdaPlayState):
    return tool_node if state.get("current_tool_calls") else memory_node

# ── Build state machine ────────────────────────────────────
machine = StateMachine[UdaPlayState](UdaPlayState)

entry        = EntryPoint[UdaPlayState]()
prepare_node = Step[UdaPlayState]("prepare",     prepare)
llm_node     = Step[UdaPlayState]("llm",         call_llm)
tool_node    = Step[UdaPlayState]("tool",         run_tool)
memory_node  = Step[UdaPlayState]("save_memory",  save_memory)
termination  = Termination[UdaPlayState]()

machine.add_steps([entry, prepare_node, llm_node, tool_node, memory_node, termination])
machine.connect(entry,        prepare_node)
machine.connect(prepare_node, llm_node)
machine.connect(llm_node,     [tool_node, memory_node], route)
machine.connect(tool_node,    llm_node)
machine.connect(memory_node,  termination)

# ── Invoke helper (with short-term memory) ─────────────────
from lib.memory import ShortTermMemory
short_term = ShortTermMemory()

def invoke_agent(query: str, session_id: str = "default") -> Run:
    short_term.create_session(session_id)
    last_run = short_term.get_last_object(session_id)
    prev_messages = last_run.get_final_state().get("messages", []) if last_run else []
    run = machine.run({
        "user_query": query,
        "instructions": INSTRUCTIONS,
        "messages": prev_messages,
        "current_tool_calls": None,
        "total_tokens": 0,
        "session_id": session_id,
    })
    short_term.add(run, session_id)
    return run

# ── Test ───────────────────────────────────────────────────
run = invoke_agent("Mario 64 release date", session_id="test")
assert run.get_final_state() is not None
print("✓ State machine agent with long-term memory ready")
print(f"✓ Nodes: {list(machine.steps.keys())}")

run1 = invoke_agent("When was Super Mario 64 released?", session_id="memory_test")
state1 = run1.get_final_state()
answer1 = next(m.content for m in reversed(state1["messages"]) if isinstance(m, AIMessage) and m.content and not m.tool_calls)
print(f"Turn 1: {answer1}\n")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: prepare
[StateMachine] Executing step: llm
[StateMachine] Executing step: tool
[StateMachine] Executing step: llm
[StateMachine] Executing step: tool
[StateMachine] Executing step: llm
[StateMachine] Executing step: save_memory
[StateMachine] Terminating: __termination__
✓ State machine agent with long-term memory ready
✓ Nodes: ['__entry__', 'prepare', 'llm', 'tool', 'save_memory', '__termination__']
[StateMachine] Starting: __entry__
[StateMachine] Executing step: prepare
[StateMachine] Executing step: llm
[StateMachine] Executing step: tool
[StateMachine] Executing step: llm
[StateMachine] Executing step: tool
[StateMachine] Executing step: llm
[StateMachine] Executing step: save_memory
[StateMachine] Terminating: __termination__
Turn 1: Super Mario 64 was released in 1996 for the Nintendo 64. It is a groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.

